## Holiday Package Prediciton

### 1) Problem statement.

Predict which customers are likely to purchase the Wellness Tourism Package using past data to reduce random marketing costs.


* The company wants to expand its customer base by introducing new travel packages.

* Currently, it offers 5 types of packages:

    Basic, Standard, Deluxe, Super Deluxe, King

* Last year’s data shows that only 18% of customers purchased any package.

* The marketing cost was high because customers were contacted randomly, without using data insights.

* To improve efficiency, the company plans to use customer data to target potential buyers more effectively.

* A new product is being launched:

    Wellness Tourism Package — travel focused on maintaining or improving health and well-being.

* The goal is to make marketing more data-driven, reduce costs, and increase package purchases.

### 2) Data Collection.

The Dataset is collected from https://www.kaggle.com/datasets/susant4learning/holiday-package-purchase-prediction

The data consists of 20 column and 4888 rows.

| Aspect              | Details                                                          |
| ------------------- | ---------------------------------------------------------------- |
| **Dataset Size**    | 4,888 rows × 20 columns                                          |
| **Type**            | Mixed (Categorical + Numerical)                                  |
| **Source**          | Kaggle                                                           |
| **Target Variable** | **ProdTaken (0 = No, 1 = Yes)**                                  |
| **Goal**            | Predict which customers are likely to purchase a holiday package |


### Customer Profile Features

| Column            | Type        | Description                                                      |
| ----------------- | ----------- | ---------------------------------------------------------------- |
| **Age**           | Numerical   | Customer's age in years                                          |
| **TypeofContact** | Categorical | How the customer was approached (Company Invited / Self Inquiry) |
| **CityTier**      | Categorical | City category (1 = Metro, 2 = Middle tier, 3 = Small cities)     |
| **Occupation**    | Categorical | Job sector of customer                                           |
| **Gender**        | Categorical | Male / Female                                                    |


### Behavioral & Interaction Features

| Column                     | Type        | Description                                        |
| -------------------------- | ----------- | -------------------------------------------------- |
| **NumberOfPersonVisiting** | Numerical   | Group size (family/friends)                        |
| **NumberOfFollowups**      | Numerical   | Number of follow-ups done before reaching decision |
| **PreferredPropertyStar**  | Numerical   | Preferred hotel star rating (3/4/5 star)           |
| **DurationOfPitch**        | Numerical   | Duration (in minutes) of the sales pitch           |
| **PitchSatisfactionScore** | Numerical   | Customer’s satisfaction with sales pitch (1–5)     |
| **ProductPitched**         | Categorical | Package type presented to the customer             |


### Customer Background & Financial Features

| Column                       | Type        | Description                                  |
| ---------------------------- | ----------- | -------------------------------------------- |
| **MaritalStatus**            | Categorical | Single / Married / Divorced                  |
| **NumberOfChildrenVisiting** | Numerical   | Number of children travelling                |
| **MonthlyIncome**            | Numerical   | Monthly income in INR                        |
| **Designation**              | Categorical | Job level (Executive, Manager, Senior, etc.) |


### Travel History

| Column        | Type        | Description                                 |
| ------------- | ----------- | ------------------------------------------- |
| **Passport**  | Categorical | 1 = Has passport, 0 = No passport           |
| **OwnCar**    | Categorical | 1 = Owns a car, 0 = No car                  |
| **ProdTaken** | Target      | 1 = Purchased package, 0 = Did not purchase |


### Target Variable Distribution

ProdTaken = 0 → Majority (Class imbalance exists)

ProdTaken = 1 → Minority


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")



df = pd.read_csv("Travel.csv")
df

## Data Cleaning


df.info()


df.isnull().sum()


df['Gender'].unique()

### Check all the categories 

df['Gender'].value_counts()


df['MaritalStatus'].unique()


df['MaritalStatus'].value_counts()


df['TypeofContact'].value_counts()


df['Gender'] = df['Gender'].replace('Fe Male', 'Female')


df['MaritalStatus'] = df['MaritalStatus'].replace('Single', 'Unmarried')

### Check all the categories 

df['MaritalStatus'].value_counts()

df.head()


df.columns

## Check Misssing Values, these are the features with nan value

features_with_na=[features for features in df.columns if df[features].isnull().sum()>=1]

features_with_na


for feature in features_with_na:
    
    print(feature,np.round(df[feature].isnull().mean()*100,2), '% missing values')
    

## Imputing Null values
1. Impute Median value for Age column
2. Impute Mode for Type of Contract
3. Impute Median for Duration of Pitch
4. Impute Mode for NumberofFollowup as it is Discrete feature
5. Impute Mode for PreferredPropertyStar
6. Impute Median for NumberofTrips
7. Impute Mode for NumberOfChildrenVisiting
8. Impute Median for MonthlyIncome

#Age
df.Age.fillna(df.Age.median(), inplace=True)

#TypeofContract
df.TypeofContact.fillna(df.TypeofContact.mode()[0], inplace=True)

#DurationOfPitch
df.DurationOfPitch.fillna(df.DurationOfPitch.median(), inplace=True)

#NumberOfFollowups
df.NumberOfFollowups.fillna(df.NumberOfFollowups.mode()[0], inplace=True)

#PreferredPropertyStar
df.PreferredPropertyStar.fillna(df.PreferredPropertyStar.mode()[0], inplace=True)

#NumberOfTrips
df.NumberOfTrips.fillna(df.NumberOfTrips.median(), inplace=True)

#NumberOfChildrenVisiting
df.NumberOfChildrenVisiting.fillna(df.NumberOfChildrenVisiting.mode()[0], inplace=True)

#MonthlyIncome
df.MonthlyIncome.fillna(df.MonthlyIncome.median(), inplace=True)


df.isnull().sum()


df.drop('CustomerID', inplace=True, axis=1)


df.shape

## Feature Engineering

### Feature Extraction


df.head()

df.dtypes

# create new column for feature

df['TotalVisiting'] = df['NumberOfPersonVisiting'] + df['NumberOfChildrenVisiting']


df.drop(columns=['NumberOfPersonVisiting', 'NumberOfChildrenVisiting'], axis=1, inplace=True)


df.dtypes, df.shape

## get all the numeric features

num_features = [feature for feature in df.columns if df[feature].dtype != 'O']

print('Num of Numerical Features :', len(num_features))

num_features 

##categorical features

cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']

print('Num of Categorical Features :', len(cat_features))

cat_features 

## Train Test Split And Model Training

from sklearn.model_selection import train_test_split

X = df.drop(['ProdTaken'], axis=1)

y = df['ProdTaken']


X.head()

y.value_counts()

X.head()

# separate dataset into train and test

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train.shape, X_test.shape

X.info()



# Create Column Transformer with 3 types of transformers

cat_features = X.select_dtypes(include="object").columns

num_features = X.select_dtypes(exclude="object").columns



from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.compose import ColumnTransformer


numeric_transformer = StandardScaler()

oh_transformer = OneHotEncoder(drop='first')


preprocessor = ColumnTransformer(
    [
         ("OneHotEncoder", oh_transformer, cat_features),
          ("StandardScaler", numeric_transformer, num_features)
    ]
)


preprocessor

## applying Trnsformation in training(fit_transform)

X_train=preprocessor.fit_transform(X_train)


pd.DataFrame(X_train)

## apply tansformation on test(transform)

X_test=preprocessor.transform(X_test)

X_test

### Adaboost Classifier Training



pd.DataFrame(X_train)

y_train

from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import GradientBoostingClassifier

from sklearn.ensemble import AdaBoostClassifier

from sklearn.metrics import accuracy_score, classification_report,ConfusionMatrixDisplay, precision_score, recall_score, f1_score, roc_auc_score,roc_curve 

import pandas as pd

models = {
    "Random Forest": RandomForestClassifier(),
    "Gradient Boost": GradientBoostingClassifier(),
    "Adaboost": AdaBoostClassifier()
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)

    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Training metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    train_precision = precision_score(y_train, y_train_pred)
    train_recall = recall_score(y_train, y_train_pred)
    train_f1 = f1_score(y_train, y_train_pred, average='weighted')
    train_rocauc = roc_auc_score(y_train, y_train_pred)

    # Testing metrics
    test_acc = accuracy_score(y_test, y_test_pred)
    test_precision = precision_score(y_test, y_test_pred)
    test_recall = recall_score(y_test, y_test_pred)
    test_f1 = f1_score(y_test, y_test_pred, average='weighted')
    test_rocauc = roc_auc_score(y_test, y_test_pred)

    # Store results
    results.append({
        'Model': name,
        'Train Accuracy': train_acc,
        'Train Precision': train_precision,
        'Train Recall': train_recall,
        'Train F1': train_f1,
        'Train ROC AUC': train_rocauc,
        'Test Accuracy': test_acc,
        'Test Precision': test_precision,
        'Test Recall': test_recall,
        'Test F1': test_f1,
        'Test ROC AUC': test_rocauc
    })

# Convert to DataFrame
results_df = pd.DataFrame(results)
results_df

results_df.set_index('Model', inplace=True)
results_df


# Display
print(results_df.sort_values('Test ROC AUC', ascending=False))

### Define Parameter Grids

# Random Forest parameter grid
rf_params = {
    "n_estimators": [100, 200, 500, 1000],
    "max_depth": [5, 10, 15, None],
    "max_features": [5, 7, "auto", "sqrt"],
    "min_samples_split": [2, 5, 10, 15]
}

# AdaBoost parameter grid
adaboost_params = {
    "n_estimators": [50, 60, 70, 80, 90, 100],
    "learning_rate": [0.5, 1.0, 1.5],
    "algorithm": ['SAMME', 'SAMME.R']
}


### Set Up Models for RandomizedSearchCV

from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import RandomizedSearchCV

# Models list for tuning
randomcv_models = [
    ("Random Forest", RandomForestClassifier(), rf_params),
    ("AdaBoost", AdaBoostClassifier(), adaboost_params)
]

randomcv_models

### Run RandomizedSearchCV

model_param = {}

for name, model, param_grid in randomcv_models:
    print(f"\nTuning {name}...")
    rand_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=10,
        scoring='roc_auc',
        cv=3,
        n_jobs=-1,
       
    )
    rand_search.fit(X_train, y_train)
    model_param[name] = rand_search.best_params_

# Display best parameters
for name, params in model_param.items():
    print(f"\n Best parameters for {name}:")
    print(params)


#### Retrain Models with Best Parameters

# Best models using best parameters
final_models = {
    "Random Forest": RandomForestClassifier(**model_param["Random Forest"]),
    "AdaBoost": AdaBoostClassifier(**model_param["AdaBoost"])
}

final_models

### Evaluate Final Tuned Models

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

for name, model in final_models.items():
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    print(f"\n===== {name} Performance =====")
    
    # Training
    print("Training Performance:")
    print("- Accuracy:", accuracy_score(y_train, y_train_pred))
    print("- Precision:", precision_score(y_train, y_train_pred))
    print("- Recall:", recall_score(y_train, y_train_pred))
    print("- F1 Score:", f1_score(y_train, y_train_pred, average='weighted'))
    print("- ROC AUC:", roc_auc_score(y_train, y_train_pred))

    # Testing
    print("\nTesting Performance:")
    print("- Accuracy:", accuracy_score(y_test, y_test_pred))
    print("- Precision:", precision_score(y_test, y_test_pred))
    print("- Recall:", recall_score(y_test, y_test_pred))
    print("- F1 Score:", f1_score(y_test, y_test_pred, average='weighted'))
    print("- ROC AUC:", roc_auc_score(y_test, y_test_pred))


from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

for name, model in final_models.items():
    y_pred = model.predict(X_test)
    print(f"\nConfusion Matrix for {name}")
    disp = ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, cmap='Blues')
    disp.ax_.set_title(f"{name} - Confusion Matrix")
    plt.grid(False)
    plt.show()


from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(8, 6))

for name, model in final_models.items():
    y_proba = model.predict_proba(X_test)[:, 1]  # Probabilities for class 1
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Tuned Models")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


### Test with Unknown Data

import pandas as pd

new_data = pd.DataFrame([{
    'Age': 35,
    'TypeofContact': 'Self Enquiry',
    'CityTier': 2,
    'DurationOfPitch': 15,
    'Occupation': 'Salaried',
    'Gender': 'Male',
    'NumberOfFollowups': 3,
    'ProductPitched': 'Deluxe',
    'PreferredPropertyStar': 4,
    'MaritalStatus': 'Married',
    'NumberOfTrips': 2,
    'Passport': 1,
    'PitchSatisfactionScore': 4,
    'OwnCar': 1,
    'NumberOfChildrenVisiting': 1,
    'Designation': 'Executive',
    'MonthlyIncome': 45000,
    'NumberOfPersonVisiting': 2
}])

new_data['TotalVisiting'] = new_data['NumberOfPersonVisiting'] + new_data['NumberOfChildrenVisiting']
new_data.drop(columns=['NumberOfPersonVisiting', 'NumberOfChildrenVisiting'], inplace=True)

new_data_processed = preprocessor.transform(new_data)

for name, model in final_models.items():
    pred = model.predict(new_data_processed)[0]
    prob = model.predict_proba(new_data_processed)[0][1]
    print(f"\n{name}:")
    print("Prediction:", "Will Purchase" if pred == 1 else "Will Not Purchase")
    print("Purchase Probability:", round(prob, 2))




Different Learning Mechanisms

    Random Forest builds many decision trees and takes a majority vote.

        It captures more complex interactions (nonlinearities, variable combinations).

AdaBoost builds models sequentially, focusing on mistakes of earlier models.

        It’s more sensitive to noise and weak patterns.

➝ So, the same input may be classified differently because the decision rules differ.

from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[('rf', final_models['Random Forest']), ('ada', final_models['AdaBoost'])],
    voting='soft'
)



voting_clf.fit(X_train, y_train)

print(voting_clf.predict(new_data_processed))


